# 00 — Sweep driver

Runs the quality gate (`sweep --check`) on this machine — meant for a **Colab GPU kernel**
(open from VSCode connected to the kernel, or in Colab directly).

- Tier `S` — nano smoke run, CPU-friendly (CI parity)
- Tier `M` — the "actually learns" bar; EvalSpec thresholds gate here (GPU)
- Tier `L` — full budgets (GPU)

The report lands in `runs/sweep/<timestamp>/report.{md,json}` and renders below.

In [ ]:
# ── Setup (works in Colab and in a local checkout) ─────────────────────────
# Colab: clone the repo into /content and pip-install it into the kernel's
# Python (honors the repo's torch/numpy pins). Locally: walk up to the repo
# root. Either way the cwd becomes the repo root so `main.py`, `data/` and
# `runs/` resolve, and `src/` + this folder are put on sys.path so imports
# work without restarting the kernel after the editable install.
import os, pathlib, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_JUPYTER_TOKEN'))
REPO_URL = 'https://github.com/juan-garassino/mini-networks.git'
if IN_COLAB:
    root = pathlib.Path('/content/mini-networks')
    if not root.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(root)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(root)], check=True)
else:
    root = pathlib.Path.cwd().resolve()
    while not (root / 'main.py').exists() and root != root.parent:
        root = root.parent
os.chdir(root)
for p in (root / 'src', root / 'colab' / 'notebooks'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
print('repo root:', root)

In [ ]:
import torch

TIER = 'M' if torch.cuda.is_available() else 'S'   # S | M | L (M/L need a GPU)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODELS = 'all'             # or comma-separated, e.g. 'gan,diffusion'
COMPOSITIONS = 'all'

print(f'tier={TIER} device={DEVICE}')

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, 'main.py', 'sweep', '--check',
    '--training_tier', TIER, '--device', DEVICE,
    '--models', MODELS, '--compositions', COMPOSITIONS,
]
proc = subprocess.run(cmd, text=True)
print('exit code:', proc.returncode)

In [ ]:
# Render the latest report inline
import pathlib
from IPython.display import Markdown, display

reports = sorted(pathlib.Path('runs/sweep').glob('*/report.md'))
display(Markdown(reports[-1].read_text()))

## Triage loop (Phase 2)

For each non-pass item in the report:

1. Diagnose — learning rate, schedule, tier budget (`core/tiers.py` overrides), or a real bug.
2. Fix in source, then re-run just that item:
   ```
   python main.py sweep --check --training_tier M --device cuda --models <name> --skip-compositions
   ```
3. Threshold changes go in `core/evalspec.py` **with a justification comment**.
4. Commit when green; note "what it took" for the docs chapter.